# 02 · Maskers: 4D brain -> 2D table  (⏱ ~75 min)

**This is the core of nilearn.** Everything later (connectivity, decoding) is built on it.

A masker is a scikit-learn-style transformer:
```
masker.fit(img)           # learn the mask / atlas geometry
X = masker.transform(img) # (n_timepoints, n_features)
img2 = masker.inverse_transform(X)  # back to a brain image
```
Three flavours:
| Masker | Features are... | Use when |
|---|---|---|
| `NiftiMasker` | every voxel in a mask | voxel-wise decoding, GLM-ish work |
| `NiftiLabelsMasker` | one signal per discrete region (atlas) | ROI-based connectivity |
| `NiftiMapsMasker` | one signal per *probabilistic* map (e.g. ICA networks) | network-level analysis |

**Cleaning options live inside the masker:** `detrend`, `standardize`, `high_pass`/`low_pass` (needs `t_r`), `confounds`, `smoothing_fwhm`. Doing it there, in one pass, avoids stacking filtering mistakes.

**Gallery to read after your attempt:** `06_manipulating_images/plot_nifti_simple.py` (NiftiMasker), `plot_mask_computation.py` (how masks are computed, incl. the `mask_strategy` options), `plot_nifti_labels_simple.py` (labels masker), and `03_connectivity/plot_signal_extraction.py` (cleaning + confounds + atlas, end to end).

In [ ]:
!pip install -q nilearn
import nilearn, numpy as np, pandas as pd, matplotlib.pyplot as plt
print("nilearn", nilearn.__version__)

**Data tip:** nilearn downloads datasets to `~/nilearn_data`, which Colab wipes when the runtime resets.
If you don't want to re-download, mount Drive and pass `data_dir=` to the `fetch_*` functions
(`from google.colab import drive; drive.mount('/content/drive')`).

In [ ]:
from nilearn import datasets, image, plotting, maskers   # older versions: nilearn.input_data
haxby = datasets.fetch_haxby(subjects=[2])
func_file = haxby.func[0]

## Exercises
- **Ex 1 (NiftiMasker).** Build a `maskers.NiftiMasker` using `mask_strategy="epi"`. Fit-transform the functional image. *Predict first*: what shape will `X` have? Print it and compare to the 4D shape. What number is the feature dimension, in terms of the mask?
- **Ex 2.** Use `masker.mask_img_` and plot it. Then take one *column* of `X`, plot it over time: that's one voxel's time series. Then use `inverse_transform` on a single row of `X` and plot it. You went brain -> table -> brain.
- **Ex 3 (cleaning).** Make two maskers, one with `standardize=False`, one with `standardize="zscore_sample"` plus `detrend=True`. Compare the time series of one voxel. What did each option change? Why does the Haxby data need `t_r=2.5` if you add `high_pass=0.01`?
- **Ex 4 (NiftiLabelsMasker).** Fetch the Harvard-Oxford (or Schaefer) atlas, use `maskers.NiftiLabelsMasker`, and get a `(time, n_regions)` matrix. How many columns? Why is this so much smaller than Ex 1?
- **Ex 5 (confounds).** `datasets.fetch_development_fmri(n_subjects=2)` returns `.func`, `.confounds` (one file per subject) and `.t_r`. Pass one subject's confounds to `transform(img, confounds=...)` and use that `t_r`. *Predict*: what should happen to the correlation between motion parameters and your signal? Verify. (Check the confounds table's columns first: which ones are motion?)
- **Ex 6 (stretch).** Read what `memory=` and `memory_level=` do on maskers. When do they save you time in a real pipeline?

In [ ]:
# Ex 1

In [ ]:
# Ex 2

In [ ]:
# Ex 3

In [ ]:
# Ex 4

In [ ]:
# Ex 5

## Check your understanding
- Why can't you just feed a 4D image straight into scikit-learn?
- When would you prefer voxel features over atlas regions, and what is the cost of each choice?
- Why is it better to regress confounds *inside* the masker than on the extracted signals afterward?

## Industry/research angle
Feature extraction pipelines: turning raw scans into ML-ready tables is how pharma and biotech imaging teams build biomarker models. The choice of atlas and cleaning options is a real, consequential analysis decision (and a classic source of "researcher degrees of freedom").